# Fake News Detector
**ML Lab Project** — TF-IDF + Random Forest Pipeline

This notebook:
1. Fetches a real fake news dataset from the web
2. Preprocesses and vectorizes text using TF-IDF
3. Trains a Random Forest classifier
4. Evaluates performance with metrics and confusion matrix
5. Provides a live prediction function for new articles

## 1. Install & Import Dependencies

In [ ]:
# Uncomment to install if needed
# !pip install pandas scikit-learn matplotlib seaborn requests nltk joblib

import pandas as pd
import numpy as np
import requests
import io
import re
import nltk
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, roc_auc_score, roc_curve
)
from sklearn.pipeline import Pipeline

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

print('All dependencies loaded successfully!')

## 2. Fetch Dataset from the Web

We use the **WELFake dataset** hosted on GitHub — a well-known benchmark for fake news detection containing ~72,000 news articles.

In [ ]:
# WELFake dataset (real/fake news benchmark)
# label: 0 = Fake, 1 = Real
DATASET_URL = "https://raw.githubusercontent.com/bhargavi1poyekar/Fake_news_Detection/main/WELFake_Dataset.csv"

print("Fetching dataset...")
try:
    response = requests.get(DATASET_URL, timeout=30)
    response.raise_for_status()
    df = pd.read_csv(io.StringIO(response.text))
    print(f"Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
except Exception as e:
    # Fallback: load from local file if you have it
    print(f"Could not fetch from URL: {e}")
    print("Tip: Download WELFake_Dataset.csv manually and place it in this folder.")
    print("Running with a small synthetic demo dataset instead...")
    # Tiny synthetic fallback for demo purposes
    data = {
        'title': [
            'Scientists confirm major climate breakthrough',
            'Aliens land in New York, government cover-up revealed',
            'New vaccine proves 95% effective in trials',
            'President secretly a lizard person says insider',
            'Stock market reaches record high amid recovery',
            'Drinking bleach cures all diseases doctors hate this',
            'UN releases annual global poverty report',
            'Moon landing was filmed in Hollywood studio'
        ] * 100,
        'text': [
            'Researchers at MIT have confirmed a new method of carbon capture.',
            'Anonymous sources claim UFOs landed near Central Park last Tuesday.',
            'Phase 3 trials published in NEJM show strong efficacy data.',
            'A whistleblower claims to have photographic proof of reptilian features.',
            'The Dow Jones rose 2.3% following positive employment data.',
            'A viral post claims miracle cure eliminates need for medicine.',
            'The United Nations released its annual report on global poverty trends.',
            'Conspiracy theorists allege Kubrick directed the 1969 moon landing footage.',
        ] * 100,
        'label': [1, 0, 1, 0, 1, 0, 1, 0] * 100
    }
    df = pd.DataFrame(data)
    print(f"Synthetic demo dataset created: {df.shape[0]} rows")

df.head()

## 3. Exploratory Data Analysis

In [ ]:
print("=== Dataset Info ===")
print(df.info())
print("\n=== Missing Values ===")
print(df.isnull().sum())
print("\n=== Label Distribution ===")
label_counts = df['label'].value_counts()
print(label_counts)
print(f"\nFake news: {label_counts.get(0, 0):,} | Real news: {label_counts.get(1, 0):,}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Label distribution
label_names = {0: 'Fake', 1: 'Real'}
counts = df['label'].map(label_names).value_counts()
axes[0].bar(counts.index, counts.values, color=['#e74c3c', '#2ecc71'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Label Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')

# Article text length distribution
text_col = 'text' if 'text' in df.columns else df.columns[1]
df['text_length'] = df[text_col].fillna('').apply(lambda x: len(str(x).split()))
for label, color, name in [(0, '#e74c3c', 'Fake'), (1, '#2ecc71', 'Real')]:
    subset = df[df['label'] == label]['text_length']
    axes[1].hist(subset.clip(upper=1000), bins=50, alpha=0.6, color=color, label=name)
axes[1].set_title('Article Length Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA plots saved to eda_plots.png')

## 4. Text Preprocessing

In [ ]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """Clean and normalize text for TF-IDF vectorization."""
    if not isinstance(text, str):
        return ''
    # Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove non-alphabetic characters
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenize, remove stopwords, stem
    tokens = text.split()
    tokens = [stemmer.stem(t) for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

# Combine title + text for richer signal
title_col = 'title' if 'title' in df.columns else df.columns[0]
text_col = 'text' if 'text' in df.columns else df.columns[1]

df['combined'] = (df[title_col].fillna('') + ' ' + df[text_col].fillna(''))
print('Preprocessing text... (this may take a moment for large datasets)')
df['cleaned'] = df['combined'].apply(preprocess_text)
print(f'Done! Sample cleaned text:\n{df["cleaned"].iloc[0][:200]}')

## 5. Train/Test Split

In [ ]:
X = df['cleaned']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples : {len(X_train):,}')
print(f'Test samples     : {len(X_test):,}')
print(f'Train label dist : {y_train.value_counts().to_dict()}')
print(f'Test label dist  : {y_test.value_counts().to_dict()}')

## 6. Build TF-IDF + Random Forest Pipeline

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),      # unigrams + bigrams
        sublinear_tf=True,       # log normalization
        min_df=2,
        max_df=0.95
    )),
    ('clf', RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1,               # use all CPU cores
        class_weight='balanced'
    ))
])

print('Training model...')
pipeline.fit(X_train, y_train)
print('Training complete!')

## 7. Evaluate the Model

In [ ]:
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
roc_auc  = roc_auc_score(y_test, y_prob)

print(f'Accuracy  : {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'ROC-AUC   : {roc_auc:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Fake', 'Real']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'])
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='#2ecc71', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)
axes[1].set_title('ROC Curve', fontsize=13, fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Evaluation plots saved to evaluation_plots.png')

## 8. Feature Importance — Top TF-IDF Terms

In [ ]:
tfidf = pipeline.named_steps['tfidf']
rf    = pipeline.named_steps['clf']
feature_names = np.array(tfidf.get_feature_names_out())
importances   = rf.feature_importances_

top_n = 20
top_idx = importances.argsort()[-top_n:][::-1]
top_features = feature_names[top_idx]
top_scores   = importances[top_idx]

plt.figure(figsize=(10, 6))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, top_n))[::-1]
bars = plt.barh(range(top_n), top_scores[::-1], color=colors[::-1])
plt.yticks(range(top_n), top_features[::-1])
plt.xlabel('Feature Importance')
plt.title(f'Top {top_n} Most Important TF-IDF Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Feature importance plot saved to feature_importance.png')

## 9. Save the Trained Model

In [ ]:
MODEL_PATH = 'fake_news_model.pkl'
joblib.dump(pipeline, MODEL_PATH)
print(f'Model saved to {MODEL_PATH}')

## 10. Live Prediction Function

Use this to test any article — paste text directly or provide a URL.

In [ ]:
def fetch_article_text(url):
    """Fetch plain text from a news URL (best-effort extraction)."""
    try:
        from urllib.request import urlopen
        from html.parser import HTMLParser

        class TextExtractor(HTMLParser):
            def __init__(self):
                super().__init__()
                self.texts = []
                self._skip = False
            def handle_starttag(self, tag, attrs):
                if tag in ('script', 'style', 'nav', 'footer', 'header'):
                    self._skip = True
            def handle_endtag(self, tag):
                if tag in ('script', 'style', 'nav', 'footer', 'header'):
                    self._skip = False
            def handle_data(self, data):
                if not self._skip and data.strip():
                    self.texts.append(data.strip())

        with urlopen(url, timeout=10) as resp:
            html = resp.read().decode('utf-8', errors='ignore')
        parser = TextExtractor()
        parser.feed(html)
        return ' '.join(parser.texts)
    except Exception as e:
        return f'Error fetching URL: {e}'


def predict_news(text=None, url=None):
    """
    Predict whether an article is real or fake.

    Parameters:
        text (str): Raw article text.
        url  (str): URL of a news article to fetch and analyze.

    Returns:
        dict with verdict, credibility score, and confidence.
    """
    if url:
        print(f'Fetching article from: {url}')
        text = fetch_article_text(url)
        if text.startswith('Error'):
            print(text)
            return None
        print(f'Fetched {len(text.split())} words.')

    if not text or len(text.split()) < 5:
        print('Error: Text is too short to analyze.')
        return None

    cleaned = preprocess_text(text)
    proba   = pipeline.predict_proba([cleaned])[0]
    fake_prob = proba[0]
    real_prob = proba[1]
    label     = 'REAL' if real_prob >= 0.5 else 'FAKE'
    credibility_score = round(real_prob * 100, 1)

    if credibility_score >= 70:
        confidence = 'High'
        emoji = '✅'
    elif credibility_score >= 45:
        confidence = 'Uncertain'
        emoji = '⚠️'
    else:
        confidence = 'High'
        emoji = '❌'

    result = {
        'verdict': label,
        'credibility_score': credibility_score,
        'confidence': confidence,
        'fake_probability': round(fake_prob * 100, 1),
        'real_probability': round(real_prob * 100, 1),
    }

    print('=' * 45)
    print(f' {emoji}  VERDICT: {label}')
    print(f'    Credibility Score : {credibility_score}/100')
    print(f'    Real probability  : {result["real_probability"]}%')
    print(f'    Fake probability  : {result["fake_probability"]}%')
    print(f'    Confidence        : {confidence}')
    print('=' * 45)
    return result

print('predict_news() function ready!')

### Try it out — paste text

In [ ]:
sample_text = """
Scientists at Oxford University have developed a new mRNA-based vaccine 
that shows 94% efficacy against the latest strain of influenza in Phase 3 trials. 
The results, published in The Lancet, involved 40,000 participants across 12 countries.
Regulatory approval is expected by Q2 next year.
"""

result = predict_news(text=sample_text)

### Try it out — from a URL

In [ ]:
# Replace with any news article URL
url = "https://www.bbc.com/news/science-environment-68886769"
result = predict_news(url=url)